# Dixie Valley aeromagnetic levelling

In [1]:
%load_ext autoreload
%autoreload 2


import logging

import cmocean
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import verde as vd

import airbornegeo

# setup logging to get some additional info from the airbornegeo functions
logging.getLogger("airbornegeo").setLevel("INFO")
logging.basicConfig()

## Load data

In [ ]:
data_df = pd.read_csv("data/san_luis_magnetics.csv")

# # every x points
# data_df = data_df[::10]

# # every x line
# data_df = data_df[
#     data_df.line.isin(data_df.line.unique()[data_df.line.unique() % 8 == 0])
# ]

# convert dataframe into geodataframe
data_df = gpd.GeoDataFrame(
    data_df,
    geometry=gpd.points_from_xy(data_df.easting, data_df.northing),
    crs="EPSG:3031",
)

data_df.head()

In [ ]:
data_df.describe()

In [ ]:
data_df["distance_along_line"] = airbornegeo.along_track_distance(
    data_df, easting_column="easting", northing_column="northing", groupby_column="line"
)
data_df.head()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 6))

df = data_df  # [::10]
df = df[df.line_type == "line"]
ax = df.plot.scatter(
    "easting",
    "northing",
    c="line",
    s=0.02,
    cmap="rainbow",
    ax=axs[0],
    colorbar=False,
    title="Lines",
)
ax.set_aspect("equal")
plt.colorbar(ax.collections[0], ax=ax, shrink=0.5)

df = data_df  # [::10]
df = df[df.line_type == "tie"]
ax = df.plot.scatter(
    "easting",
    "northing",
    c="line",
    s=0.02,
    cmap="rainbow",
    ax=axs[1],
    colorbar=False,
    title="Ties",
)
ax.set_aspect("equal")
plt.colorbar(ax.collections[0], ax=ax, shrink=0.5)

plt.tight_layout()
plt.show()

# Cross-over levelling

## Find intersections

In [ ]:
# calculate theoretical intersection points
inters = airbornegeo.create_intersection_table(
    data_df,
)
inters

## Add intersections as rows to the dataframe

In [ ]:
data_df.head()

In [ ]:
data_df, inters = airbornegeo.interpolate_intersections(
    data_df,
    inters,
    to_interp="unlevelled_tfa",
    window_width=500,
    method="cubic",
    extrapolate=False,
)

In [ ]:
# see which lines don't have intersections
airbornegeo.lines_without_intersections(data_df, inters)

## Calculate initial cross-over errors

In [ ]:
inters = airbornegeo.calculate_crossover_errors(
    data_df,
    inters,
    data_col="unlevelled_tfa",
    plot_map=False,
)

In [ ]:
data_df, inters_alternating_iterative = (
    airbornegeo.alternating_iterative_line_levelling(
        data_df,
        inters,
        data_col="unlevelled_tfa",
        levelled_col="mag_levelled_alternating_trend1",
        degree=1,
        max_iterations=5,
        rms_percent_change_tolerance=25,
        plot_dynamic_levelling_convergence=True,
    )
)
inters_alternating_iterative.head()

In [ ]:
airbornegeo.plot_levelling_convergence(inters_alternating_iterative)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 40))

max_abs = vd.maxabs(data_df.unlevelled_tfa, percentile=95)
ax = data_df.plot.scatter(
    "easting",
    "northing",
    c="unlevelled_tfa",
    s=0.1,
    ax=axs[0],
    cmap=cmocean.cm.balance,
    vmin=-max_abs,
    vmax=max_abs,
    colorbar=False,
    title="Unlevelled",
)
ax.set_aspect("equal")
plt.colorbar(ax.collections[0], ax=ax, shrink=0.1)

data_df["levelling_correction"] = (
    data_df.mag_levelled_alternating_trend1 - data_df.unlevelled_tfa
)
max_abs_level_corr = vd.maxabs(data_df.levelling_correction, percentile=99.5)
ax = data_df.plot.scatter(
    "easting",
    "northing",
    c="levelling_correction",
    s=0.1,
    ax=axs[1],
    cmap=cmocean.cm.balance,
    vmin=-max_abs_level_corr,
    vmax=max_abs_level_corr,
    colorbar=False,
    title="Levelling correction",
)
ax.set_yticks([])
ax.set_aspect("equal")
plt.colorbar(ax.collections[0], ax=ax, shrink=0.1)

ax = data_df.plot.scatter(
    "easting",
    "northing",
    c="mag_levelled_alternating_trend1",
    s=0.1,
    ax=axs[2],
    cmap=cmocean.cm.balance,
    vmin=-max_abs,
    vmax=max_abs,
    colorbar=False,
    title="Alternating iterative levelling, trend 1",
)
ax.set_yticks([])
ax.set_aspect("equal")
plt.colorbar(ax.collections[0], ax=ax, shrink=0.1)

plt.tight_layout()
plt.show()

In [ ]:
data_df.levelling_correction.plot.hist(bins=100)

In [ ]:
data_df